# PART 3.2: Word2Vec

In [ ]:
import os, sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import nltk

nltk.download("punkt")
nltk.download("punkt_tab")

sys.path.append(os.path.abspath(os.path.join("..", "..")))

from collections import defaultdict, Counter
from dotenv import load_dotenv

from myapp.search import load_corpus as lc
from project_progress.part_1.data_prep import (
    corpus_df_loading,
    build_terms,
    join_build_terms,
)
from project_progress.part_2.index_tf_idf import create_index_tf_idf

from gensim.models import Word2Vec
from gensim.utils import simple_preprocess
from gensim.parsing.preprocessing import preprocess_string


load_dotenv()  # take environment variables from .env

[nltk_data] Downloading package stopwords to /home/nara-
[nltk_data]     dellans/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

Functions to store the time-consuming processing in files to load faster after:

In [ ]:
products_filepath = "../../data/products.json"
products_numeric_data_filepath = "../../data/products_numeric_data.json"


def dump_data(data, filepath):
    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


def load_data(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        data = json.load(f)
    return data


def get_or_create(filepath, compute_data_function):
    if os.path.exists(filepath):
        with open(filepath, "r", encoding="utf-8") as f:
            return json.load(f)
    else:
        data = compute_data_function()
        with open(filepath, "w+", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=4)
        return data

In [ ]:
# def process_products(corpus):
#     """
#     Function that loads the products of the corpus as a dictionary of pid -> list (of categorical data as a list of processed tokens)

#     :param corpus: Corpus with all the products and all their data.
#     :return products: (dict) pid -> tokens (all preprocessed terms of the categorical data)
#     """
#     products_sentences = {}
#     for product in list(corpus.values()):

#         # Process the documents categorical fields as in the creation of the inverted index, but concatenate all the terms in a single string (in this case we do not care about the fields)
#         subgroups = [
#             [product.title, product.description],
#             [product.brand, product.category, product.sub_category],
#             [
#                 product.seller,
#                 " ".join([detail for detail in product.product_details.values()]),
#             ],
#         ]

#         products_sentences[product.pid] = [
#             join_build_terms(group) for group in subgroups
#         ]

#     return products_sentences

In [56]:
def process_products(corpus, preprocess=preprocess_string):
    """
    Function that loads the products of the corpus as a dictionary of pid -> list (of categorical data as a list of processed tokens)

    :param corpus: Corpus with all the products and all their data.
    :return products: (dict) pid -> tokens (all preprocessed terms of the categorical data)
    """
    products_sentences = {}
    for product in list(corpus.values()):

        # Process the documents categorical fields as in the creation of the inverted index, but concatenate all the terms in a single string (in this case we do not care about the fields)
        subgroups = [
            " ".join([product.title, product.description]),
            " ".join([product.brand, product.category, product.sub_category]),
            " ".join([product.seller, 
                      " ".join([detail for detail in product.product_details.values()])]
                      )
            ]
        products_sentences[product.pid] = [preprocess(group) for group in subgroups]

    return products_sentences

Now, we load the corpus:

In [ ]:
json_path = "../../data/fashion_products_dataset.json"
corpus = corpus_df_loading(json_path)

In [101]:
# Preprocess the corpus to get the products
product_sentences = get_or_create(products_filepath, lambda: process_products(corpus))

## Document Representation using Word2Vec 

In [ ]:
sentence_list = list()
for doc_sen in product_sentences.values():
    sentence_list.extend(doc_sen)

In [36]:
w2v_model = Word2Vec(
    sentences=sentence_list, vector_size=100, window=7, min_count=5, negative=10, sg=1
)

In [ ]:
def get_doc_embedding(doc_terms, model:Word2Vec, preprocess=preprocess_string):
    
    word_embeddings = [model.wv[word] for word in doc_terms]
    embedding = np.mean(word_embeddings, axis=0)
    return embedding

In [ ]:
def build_doc_terms(product_sentences):
    product_terms = defaultdict(list)

    for pid, sentences in product_sentences.items():    
        product_terms[pid] = []
        for sentence in sentences:
            product_terms[pid].extend(sentence)

    return product_terms

In [ ]:
queries = [
    "western leather jacket men",  # context, material, specific cloth, gender
    "cotton innerwear man",  # material, specific cloth, gender
    "yellow black t-shirt women xl",  # adjectives, specific cloth, gender, size
    "casual comfortable blue trousers women",  # context, adjectives, specific cloth, gender
    "breathable sports clothes winter",
]  # adjectives, context, general clothes, context

In [108]:
product_terms = build_doc_terms(product_sentences)

AttributeError: 'dict' object has no attribute 'extend'

In [71]:
type(product_sentences["TKPFCZ9EA7H5FYZH"])

list

In [106]:
for elem in product_terms.items():
    print(elem)
    break

('TKPFCZ9EA7H5FYZH', [['solid', 'women', 'multicolor', 'track', 'pant', 'yorker', 'trackpant', 'made', '100', 'rich', 'comb', 'cotton', 'give', 'rich', 'friendli', 'waistband', 'great', 'year', 'round', 'use', 'proudli', 'made', 'india'], ['york', 'cloth', 'accessori', 'bottomwear'], ['shyam', 'enterpris', '1005combo2', 'elast', 'side', 'pocket', 'cotton', 'blend', 'solid', 'multicolor']])


In [42]:
w2v_model.wv["breath"].shape

(100,)